In [3]:
import requests

def search_wikipedia(query: str):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json"
    }

    headers = {
        "User-Agent": "AIEngineeringBuildcamp/1.0"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers
    )

    response.raise_for_status()

    return response.json()

In [4]:
result = search_wikipedia("capybara")

search_results = result["query"]["search"]

len(search_results)

10

In [5]:
titles = [item["title"] for item in search_results]

matching_titles = [
    title
    for title in titles
    if "capybara" in title.lower()
]

titles, matching_titles, len(matching_titles)

(['Capybara',
  'Lesser capybara',
  'Capybara Games',
  'Capybara (disambiguation)',
  'Capybara Party',
  'Capybara (software)',
  'Hydrochoerus',
  'Caviidae',
  'Flow (2024 film)',
  'Yuzu bath'],
 ['Capybara',
  'Lesser capybara',
  'Capybara Games',
  'Capybara (disambiguation)',
  'Capybara Party',
  'Capybara (software)'],
 6)

In [6]:
def get_page(title: str):
    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": True,
        "titles": title,
        "format": "json"
    }

    headers = {
        "User-Agent": "AIEngineeringBuildcamp/1.0"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers
    )

    response.raise_for_status()

    data = response.json()
    pages = data["query"]["pages"]

    page = next(iter(pages.values()))

    return page["extract"]

In [7]:
content = get_page("Capybara")

len(content)

15526

In [8]:
tools = [
    search_wikipedia,
    get_page
]

In [9]:
from pydantic_ai import Agent

wikipedia_agent = Agent(
    name="wikipedia",
    model="openai:gpt-4o-mini",
    instructions="""
You are a Wikipedia research assistant.

Use search_wikipedia when you need to find relevant Wikipedia pages.

Use get_page to fetch the full content of a specific Wikipedia page.

Answer using information from the Wikipedia pages you retrieve.
""",
    tools=tools
)

In [10]:
result = await wikipedia_agent.run(
    "What is this page about? https://en.wikipedia.org/wiki/Capybara"
)

result.output

'The Wikipedia page on **Capybara** provides extensive information about this unique animal. Here are the key points:\n\n- **Species Overview**: The capybara is the largest living rodent, native to almost all of South America except for Chile. It is semiaquatic and primarily herbivorous, living near freshwater bodies and feeding on grasses and aquatic plants.\n\n- **Etymology**: The name "capybara" comes from the Tupi word "ka\'apiûara," meaning "one who eats slender leaves." Its scientific genus name, **Hydrochoerus**, and species name, **Hydrochoerus hydrochaeris**, derive from Greek words meaning "water" and "pig."\n\n- **Social Behavior**: Capybaras are social animals, typically found in groups of 10-20 but can congregate in larger groups during dry seasons. They communicate using various vocalizations like barks and whistles.\n\n- **Diet and Ecology**: They are selective grazers and also practice coprophagy, eating their feces to extract maximum nutrients. Capybaras thrive in habi

In [11]:
from pydantic_ai.messages import FunctionToolCallEvent

class NamedCallback:

    def __init__(self, agent):
        self.agent_name = agent.name

    async def print_function_calls(self, ctx, event):
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)

In [12]:
callback = NamedCallback(wikipedia_agent)

result = await wikipedia_agent.run(
    "What are the main threats to capybara populations?",
    event_stream_handler=callback
)

TOOL CALL (wikipedia): search_wikipedia({"query":"capybara threats population decline"})
TOOL CALL (wikipedia): get_page({"title":"Capybara"})


In [13]:
result.output

'The main threats to capybara populations include:\n\n1. **Hunting**: Capybaras are hunted for their meat and pelts in various regions of South America. Hunting can significantly reduce population numbers in certain areas.\n\n2. **Habitat Loss**: Urbanization and agricultural expansion can lead to the destruction of their natural habitats. Capybaras depend on wetlands and proximity to bodies of water, which are often altered or drained for development.\n\n3. **Human-Wildlife Conflict**: In some regions, capybaras are seen as competition for livestock, leading to them being killed by farmers.\n\n4. **Predation**: While not a direct threat to their overall population stability, predation by jaguars, ocelots, and caimans does affect local population dynamics.\n\nDespite these threats, capybara populations are generally stable across much of their range, thanks to their adaptability to human-altered landscapes and rapid breeding capabilities.'